In [2]:

from __future__ import annotations

import copy
import math
import random
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")

# ---- repo root ----
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find repo root containing /src")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)
print("Torch        =", torch.__version__)
print("CUDA         =", torch.cuda.is_available())

from src.evaluation import evaluate_binary_probabilities
from src.models.feature_sets import (
    BASE_TABULAR_FEATURES,
    GCMT_FEATURES,
    QUALITY_FEATURES,
)
from src.models.input_layer import (
    InputConfig,
    load_modeling_splits,
    prepare_tabular_inputs,
)
from src.utils.paths import METRICS_DIR


PROJECT_ROOT = /Users/tonieenriquez/Desktop/SUTD/SoftwareConstruction/CDS_Group10
Torch        = 2.10.0
CUDA         = False


In [3]:

DATASET_NAME = "earthquake_aftershock_v2_gcmt"
MODEL_NAME = "mlp_best_v1"
TARGETS = ["y_24h", "y_72h"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)


In [4]:

splits = load_modeling_splits(dataset_name=DATASET_NAME)

for split_name, df in splits.items():
    print(
        f"{split_name:5s}  {len(df):>5} rows  |  "
        f"y_24h pos: {df['y_24h'].mean():.3f}  "
        f"y_72h pos: {df['y_72h'].mean():.3f}"
    )


train  23031 rows  |  y_24h pos: 0.443  y_72h pos: 0.504
val     1744 rows  |  y_24h pos: 0.478  y_72h pos: 0.535
test    3513 rows  |  y_24h pos: 0.469  y_72h pos: 0.530


In [5]:

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    has_gcmt = df["has_gcmt"].fillna(False) if "has_gcmt" in df.columns else pd.Series(False, index=df.index)

    # Depth regime
    if "trigger_depth_km" in df.columns:
        df["depth_shallow"] = (df["trigger_depth_km"] < 70).astype(float)
        df["depth_intermediate"] = (df["trigger_depth_km"].between(70, 300)).astype(float)
        df["depth_deep"] = (df["trigger_depth_km"] > 300).astype(float)

    # Magnitude / activity transforms
    if "trigger_magnitude" in df.columns:
        df["log_magnitude"] = np.log10(df["trigger_magnitude"].clip(lower=1e-6))

    if "trigger_magnitude" in df.columns and "trigger_depth_km" in df.columns:
        df["mag_depth_ratio"] = df["trigger_magnitude"] / (df["trigger_depth_km"] + 1)

    if "prior_global_event_count_24h" in df.columns:
        df["log_prior_24h"] = np.log1p(df["prior_global_event_count_24h"])

    if "prior_global_event_count_7d" in df.columns:
        df["log_prior_7d"] = np.log1p(df["prior_global_event_count_7d"])

    if "prior_global_event_count_24h" in df.columns and "prior_global_event_count_7d" in df.columns:
        denom = (df["prior_global_event_count_7d"] / 7.0).replace(0, np.nan)
        df["seismicity_acceleration"] = df["prior_global_event_count_24h"] / denom

    if "trigger_magnitude" in df.columns and "prior_global_event_count_24h" in df.columns:
        df["mag_x_log_prior"] = df["trigger_magnitude"] * np.log1p(df["prior_global_event_count_24h"])

    if "trigger_magnitude" in df.columns and "depth_shallow" in df.columns:
        df["mag_x_shallow"] = df["trigger_magnitude"] * df["depth_shallow"]

    # Time cyclicals
    if "trigger_month" in df.columns:
        df["sin_month"] = np.sin(2 * np.pi * df["trigger_month"] / 12)
        df["cos_month"] = np.cos(2 * np.pi * df["trigger_month"] / 12)

    if "trigger_hour" in df.columns:
        df["sin_hour"] = np.sin(2 * np.pi * df["trigger_hour"] / 24)
        df["cos_hour"] = np.cos(2 * np.pi * df["trigger_hour"] / 24)

    if "trigger_dayofyear" in df.columns:
        df["sin_dayofyear"] = np.sin(2 * np.pi * df["trigger_dayofyear"] / 365)
        df["cos_dayofyear"] = np.cos(2 * np.pi * df["trigger_dayofyear"] / 365)

    # Spatial
    if "trigger_longitude" in df.columns and "trigger_latitude" in df.columns:
        df["ring_of_fire"] = (
            (np.abs(df["trigger_longitude"]) > 130)
            & (df["trigger_latitude"].between(-60, 60))
        ).astype(float)

    # GCMT-enriched features
    if "gcmt_scalar_moment" in df.columns:
        df["log_scalar_moment"] = np.where(
            has_gcmt & df["gcmt_scalar_moment"].notna(),
            np.log10(df["gcmt_scalar_moment"].clip(lower=1e-10)),
            np.nan,
        )

    if "gcmt_moment_exponent" in df.columns:
        df["moment_exponent_centered"] = (df["gcmt_moment_exponent"] - 24.0).where(has_gcmt)

    if "gcmt_scalar_moment" in df.columns and "trigger_magnitude" in df.columns:
        df["gcmt_mw"] = np.where(
            has_gcmt & df["gcmt_scalar_moment"].notna(),
            (2 / 3) * np.log10(df["gcmt_scalar_moment"].clip(lower=1e-10)) - 10.7,
            np.nan,
        )
        df["mw_trigger_diff"] = (df["gcmt_mw"] - df["trigger_magnitude"]).where(has_gcmt)

    if "dip" in df.columns:
        df["sin_dip"] = np.sin(np.radians(df["dip"].where(has_gcmt)))

    if {"gcmt_eig1", "gcmt_eig2", "gcmt_eig3"}.issubset(df.columns):
        e1 = df["gcmt_eig1"].where(has_gcmt)
        e2 = df["gcmt_eig2"].where(has_gcmt)
        e3 = df["gcmt_eig3"].where(has_gcmt)
        denom = (e1.abs() + e3.abs()).replace(0, np.nan)
        df["clvd_fraction"] = (2 * e2.abs() / denom).where(has_gcmt)
        df["eig_ratio"] = (e1.abs() / denom).where(has_gcmt)

    if {"gcmt_eig1_plunge", "gcmt_eig3_plunge"}.issubset(df.columns):
        p1 = df["gcmt_eig1_plunge"].where(has_gcmt)
        p3 = df["gcmt_eig3_plunge"].where(has_gcmt)
        df["sin_eig1_plunge"] = np.sin(np.radians(p1))
        df["cos_eig1_plunge"] = np.cos(np.radians(p1))
        df["sin_eig3_plunge"] = np.sin(np.radians(p3))
        df["cos_eig3_plunge"] = np.cos(np.radians(p3))
        df["tp_plunge_diff"] = (p1 - p3).where(has_gcmt)

    if "gcmt_depth_km" in df.columns and "trigger_depth_km" in df.columns:
        df["centroid_depth_diff"] = (df["gcmt_depth_km"] - df["trigger_depth_km"]).where(has_gcmt)

    if "gcmt_half_duration_sec" in df.columns:
        df["log_half_duration"] = np.log1p(df["gcmt_half_duration_sec"].where(has_gcmt))

    if "gcmt_mag_diff" in df.columns:
        df["mag_diff_abs"] = df["gcmt_mag_diff"].abs().where(has_gcmt)

    if "rake" in df.columns:
        def _rake_regimes(rake: float):
            if pd.isna(rake):
                return np.nan, np.nan, np.nan
            r = rake % 360
            ss_dist = min(abs(r), abs(r - 180), abs(r - 360))
            return float(ss_dist < 45), float(45 <= r <= 135), float(225 <= r <= 315)

        regimes = df["rake"].apply(_rake_regimes)
        df["is_strike_slip"] = regimes.apply(lambda x: x[0]).where(has_gcmt)
        df["is_reverse"] = regimes.apply(lambda x: x[1]).where(has_gcmt)
        df["is_normal"] = regimes.apply(lambda x: x[2]).where(has_gcmt)

    return df

splits_eng = {name: engineer_features(df) for name, df in splits.items()}
print("Feature engineering complete.")
print("Columns after engineering:", splits_eng["train"].shape[1])


Feature engineering complete.
Columns after engineering: 127


In [6]:

ENGINEERED_FEATURES = [
    "depth_shallow", "depth_intermediate", "depth_deep",
    "log_magnitude", "mag_depth_ratio",
    "log_prior_24h", "log_prior_7d", "seismicity_acceleration",
    "mag_x_log_prior", "mag_x_shallow",
    "sin_month", "cos_month",
    "sin_hour", "cos_hour",
    "sin_dayofyear", "cos_dayofyear",
    "ring_of_fire",
    "log_scalar_moment", "moment_exponent_centered",
    "gcmt_mw", "mw_trigger_diff",
    "sin_dip",
    "clvd_fraction", "eig_ratio",
    "sin_eig1_plunge", "cos_eig1_plunge",
    "sin_eig3_plunge", "cos_eig3_plunge",
    "tp_plunge_diff",
    "centroid_depth_diff",
    "log_half_duration",
    "mag_diff_abs",
    "is_strike_slip", "is_reverse", "is_normal",
]

_seen = set()
FULL_FEATURE_SET = []
for f in (BASE_TABULAR_FEATURES + QUALITY_FEATURES + GCMT_FEATURES + ENGINEERED_FEATURES):
    if f not in _seen:
        FULL_FEATURE_SET.append(f)
        _seen.add(f)

missing_engineered = [c for c in ENGINEERED_FEATURES if c not in splits_eng["train"].columns]
print("Total requested features:", len(FULL_FEATURE_SET))
print("Missing engineered features in train:", missing_engineered)


Total requested features: 91
Missing engineered features in train: []


In [7]:

prepared = {}

for target in TARGETS:
    config = InputConfig(
        feature_cols=FULL_FEATURE_SET,
        target_col=target,
        missing_strategy="median",
        scale=True,
        allow_missing_optional=True,
        drop_rows_with_missing_target=True,
    )
    inputs = prepare_tabular_inputs(config=config, splits=splits_eng)
    prepared[target] = inputs

    print(
        f"{target} | train={inputs.X_train.shape}, val={inputs.X_val.shape}, test={inputs.X_test.shape}"
    )


y_24h | train=(23031, 91), val=(1744, 91), test=(3513, 91)
y_72h | train=(23031, 91), val=(1744, 91), test=(3513, 91)


In [8]:

class TabularMLP(nn.Module):
    def __init__(self, in_features: int, hidden_dims=(256, 128, 64), dropout=0.20):
        super().__init__()
        layers = []
        prev = in_features
        for h in hidden_dims:
            layers.extend([
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev = h
        self.backbone = nn.Sequential(*layers)
        self.head = nn.Linear(prev, 1)

        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.backbone(x)
        return self.head(x).squeeze(1)


class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_temp = nn.Parameter(torch.zeros(1))

    def forward(self, logits):
        temperature = torch.exp(self.log_temp).clamp(min=1e-3, max=100.0)
        return logits / temperature


def make_loaders(X_train, y_train, X_val, y_val, batch_size=512):
    X_train_t = torch.tensor(X_train.values, dtype=torch.float32)
    y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
    X_val_t = torch.tensor(X_val.values, dtype=torch.float32)
    y_val_t = torch.tensor(y_val.values, dtype=torch.float32)

    train_loader = DataLoader(
        TensorDataset(X_train_t, y_train_t),
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
    )
    val_loader = DataLoader(
        TensorDataset(X_val_t, y_val_t),
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
    )
    return train_loader, val_loader


@torch.no_grad()
def predict_logits(model, X, batch_size=2048, device=DEVICE):
    model.eval()
    X_t = torch.tensor(X.values, dtype=torch.float32)
    loader = DataLoader(TensorDataset(X_t), batch_size=batch_size, shuffle=False)
    all_logits = []
    for (xb,) in loader:
        xb = xb.to(device)
        logits = model(xb)
        all_logits.append(logits.detach().cpu())
    return torch.cat(all_logits).numpy()


def fit_temperature_on_val(val_logits, y_val, max_iter=300):
    scaler = TemperatureScaler().to(DEVICE)
    logits_t = torch.tensor(val_logits, dtype=torch.float32, device=DEVICE)
    y_t = torch.tensor(y_val.values, dtype=torch.float32, device=DEVICE)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.LBFGS(scaler.parameters(), lr=0.05, max_iter=max_iter)

    def closure():
        optimizer.zero_grad()
        loss = criterion(scaler(logits_t), y_t)
        loss.backward()
        return loss

    optimizer.step(closure)
    return scaler


def logits_to_prob(logits, temperature_scaler=None):
    logits_t = torch.tensor(logits, dtype=torch.float32)
    if temperature_scaler is not None:
        with torch.no_grad():
            logits_t = temperature_scaler(logits_t.to(DEVICE)).cpu()
    return torch.sigmoid(logits_t).numpy()


In [10]:

def train_one_target(
    inputs,
    seed=42,
    hidden_dims=(256, 128, 64),
    dropout=0.20,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=512,
    max_epochs=300,
    patience=30,
):
    seed_everything(seed)

    train_loader, val_loader = make_loaders(
        inputs.X_train, inputs.y_train,
        inputs.X_val, inputs.y_val,
        batch_size=batch_size,
    )

    model = TabularMLP(
        in_features=inputs.X_train.shape[1],
        hidden_dims=hidden_dims,
        dropout=dropout,
    ).to(DEVICE)

    pos_rate = float(inputs.y_train.mean())
    pos_weight_value = (1.0 - pos_rate) / max(pos_rate, 1e-8)
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([pos_weight_value], dtype=torch.float32, device=DEVICE)
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=8,
        min_lr=1e-5,
    )

    best_state = None
    best_val_logloss = math.inf
    best_epoch = -1
    wait = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            train_losses.append(loss.item())

        val_logits = predict_logits(model, inputs.X_val)
        val_prob = logits_to_prob(val_logits)
        val_metrics = evaluate_binary_probabilities(inputs.y_val, val_prob)
        val_logloss = val_metrics["log_loss"]
        scheduler.step(val_logloss)

        history.append({
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)),
            "val_log_loss": val_metrics["log_loss"],
            "val_brier_score": val_metrics["brier_score"],
            "val_roc_auc": val_metrics["roc_auc"],
            "lr": optimizer.param_groups[0]["lr"],
        })

        improved = val_logloss < best_val_logloss - 1e-5
        if improved:
            best_val_logloss = val_logloss
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if epoch % 10 == 0 or epoch == 1:
            print(
                f"Epoch {epoch:03d} | train_loss={np.mean(train_losses):.4f} "
                f"| val_logloss={val_metrics['log_loss']:.4f} "
                f"| val_brier={val_metrics['brier_score']:.4f} "
                f"| val_auc={val_metrics['roc_auc']:.4f} "
                f"| lr={optimizer.param_groups[0]['lr']:.6f}"
            )

        if wait >= patience:
            print(f"Early stopping at epoch {epoch}. Best epoch was {best_epoch}.")
            break

    model.load_state_dict(best_state)

    val_logits = predict_logits(model, inputs.X_val)
    temp_scaler = fit_temperature_on_val(val_logits, inputs.y_val)

    return model, temp_scaler, pd.DataFrame(history), {
        "best_epoch": best_epoch,
        "best_val_logloss": best_val_logloss,
        "pos_weight": pos_weight_value,
    }


In [11]:

trained = {}

MLP_PARAMS = {
    "hidden_dims": (256, 128, 64),
    "dropout": 0.20,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "batch_size": 512,
    "max_epochs": 300,
    "patience": 30,
}

for target in TARGETS:
    print(f"\n===== Training {target} =====")
    model, temp_scaler, history_df, info = train_one_target(prepared[target], **MLP_PARAMS)
    trained[target] = {
        "model": model,
        "temp_scaler": temp_scaler,
        "history": history_df,
        "info": info,
    }
    print(history_df.tail())
    print(info)



===== Training y_24h =====
Epoch 001 | train_loss=0.7168 | val_logloss=0.5329 | val_brier=0.1799 | val_auc=0.8081 | lr=0.001000
Epoch 010 | train_loss=0.5829 | val_logloss=0.5176 | val_brier=0.1734 | val_auc=0.8212 | lr=0.001000
Epoch 020 | train_loss=0.5496 | val_logloss=0.5249 | val_brier=0.1756 | val_auc=0.8182 | lr=0.001000
Epoch 030 | train_loss=0.5219 | val_logloss=0.5218 | val_brier=0.1735 | val_auc=0.8193 | lr=0.000500
Epoch 040 | train_loss=0.5045 | val_logloss=0.5297 | val_brier=0.1760 | val_auc=0.8154 | lr=0.000250
Early stopping at epoch 49. Best epoch was 19.
    epoch  train_loss  val_log_loss  val_brier_score  val_roc_auc        lr
44     45    0.499796      0.530339         0.176039     0.814400  0.000250
45     46    0.496023      0.531700         0.176427     0.814595  0.000125
46     47    0.495550      0.532834         0.176873     0.813694  0.000125
47     48    0.497970      0.532583         0.176609     0.814573  0.000125
48     49    0.495728      0.531767     

In [12]:

summary_rows = []
prediction_tables = []

for target in TARGETS:
    inp = prepared[target]
    model = trained[target]["model"]
    temp_scaler = trained[target]["temp_scaler"]
    horizon = int(target.split("_")[1].replace("h", ""))

    split_map = {
        "train": (inp.X_train, inp.y_train, splits_eng["train"]),
        "val": (inp.X_val, inp.y_val, splits_eng["val"]),
        "test": (inp.X_test, inp.y_test, splits_eng["test"]),
    }

    for split_name, (X_df, y_s, raw_df) in split_map.items():
        logits = predict_logits(model, X_df)
        y_prob_uncal = logits_to_prob(logits)
        y_prob = logits_to_prob(logits, temperature_scaler=temp_scaler)

        metrics = evaluate_binary_probabilities(y_s, y_prob)
        metrics_uncal = evaluate_binary_probabilities(y_s, y_prob_uncal)

        summary_rows.append({
            "model_name": MODEL_NAME,
            "split": split_name,
            "horizon": horizon,
            "calibrated": True,
            **metrics,
        })
        summary_rows.append({
            "model_name": MODEL_NAME + "_uncalibrated",
            "split": split_name,
            "horizon": horizon,
            "calibrated": False,
            **metrics_uncal,
        })

        prediction_tables.append(pd.DataFrame({
            "trigger_event_id": raw_df["trigger_event_id"].values,
            "split": split_name,
            "horizon": horizon,
            "model_name": MODEL_NAME,
            "y_true": y_s.values,
            "y_prob": y_prob,
        }))

metrics_df = pd.DataFrame(summary_rows)
predictions_df = pd.concat(prediction_tables, ignore_index=True)

print(metrics_df.sort_values(["horizon", "calibrated", "split"]))
print(predictions_df.head())


                  model_name  split  horizon  calibrated  brier_score  \
5   mlp_best_v1_uncalibrated   test       24       False     0.178912   
1   mlp_best_v1_uncalibrated  train       24       False     0.149710   
3   mlp_best_v1_uncalibrated    val       24       False     0.171671   
4                mlp_best_v1   test       24        True     0.178742   
0                mlp_best_v1  train       24        True     0.149956   
2                mlp_best_v1    val       24        True     0.171635   
11  mlp_best_v1_uncalibrated   test       72       False     0.194787   
7   mlp_best_v1_uncalibrated  train       72       False     0.158768   
9   mlp_best_v1_uncalibrated    val       72       False     0.183597   
10               mlp_best_v1   test       72        True     0.193806   
6                mlp_best_v1  train       72        True     0.159628   
8                mlp_best_v1    val       72        True     0.183405   

    log_loss   roc_auc  n_obs  positive_rate  
5  

In [13]:

METRICS_DIR.mkdir(parents=True, exist_ok=True)

metrics_path = METRICS_DIR / f"{MODEL_NAME}_metrics.csv"
preds_path = METRICS_DIR / f"{MODEL_NAME}_predictions.csv"

metrics_df.to_csv(metrics_path, index=False)
predictions_df.to_csv(preds_path, index=False)

print("Saved:", metrics_path)
print("Saved:", preds_path)


Saved: /Users/tonieenriquez/Desktop/SUTD/SoftwareConstruction/CDS_Group10/reports/metrics/mlp_best_v1_metrics.csv
Saved: /Users/tonieenriquez/Desktop/SUTD/SoftwareConstruction/CDS_Group10/reports/metrics/mlp_best_v1_predictions.csv
